# Evaluate saved charts and a saved CNN-LSTM run

Notebook 03 trains and saves; this notebook never trains, re-extracts features or regenerates splits.
All tunable evaluation settings live in the single `CONFIG` dict in
`src/donderformer/evaluation_config.py`, each marked **PLACEHOLDER - team to confirm**.
Review that dict before running; the notebook copies it, and writes settings with its results.
Set `RUN_DIR` to an existing saved run. Inference uses its saved data/model configuration.
`DataPaths()` chooses `DONDER_DATA` or `<repo>/data`.

For scoring **already saved charts from any model**, no checkpoint or mel features are needed:
use the model-independent example at the end. You do need the original train and test
cached labels and metadata for training bands and pooled human pattern references.
Missing files produce actionable errors; none are generated automatically.

Difficulty is merged star level within Oni. The manifest has blank levels, so LEVEL is read from the matching raw TJA section without rewriting the split.

Existing exporter files stay at `eval/ar_test/charts/<chart_id>.npz` with `human`,
`generated`, `clean`. Existing results.json and summary.csv retain their schemas.
New multi-window metrics, all/difficulty summaries, configuration and training bands
are saved in `eval/ar_test/analysis/`; random-baseline results are inside its
`random_baseline/` folder. Teacher-forced charts are additionally saved under
`eval/teacher_forced/charts/`, and their new scoring outputs under `teacher_forced/scored/`.
Existing chart files are reused for AR resume; move stale chart files aside when changing checkpoints.


## Oni-only difficulty groups and sparse references

This dataset contains **Oni charts only**. Here, "difficulty" means **star level within Oni**.
The manifest's blank LEVEL values are recovered from the corresponding raw TJA course.
`MIN_CHARTS_PER_BAND_GROUP=30` is the provisional minimum number of training charts per group.
We merge the lowest sparse star group with its nearest adjacent group, repeat as needed,
and break equal-distance ties toward the lower-star neighbor. Only training counts choose
merges; test counts are reported, not used to choose them.

The merged mapping is shared by training bands/type distributions, pooled HI references,
and every difficulty summary. Training/test counts print below and are saved in each
`analysis/difficulty_groups.json` and `.csv`, together with original counts and merge history.
If even all available training charts cannot meet the threshold, the single remaining group
is explicitly marked below minimum. Counts exclude charts in the matching extractor failure list.


In [ ]:
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

from donderformer.paths import DataPaths, outputs_dir
from donderformer.evaluation_config import evaluation_config
from donderformer.chart_evaluation import load_reference_context, evaluate_random_baseline
from donderformer.evaluation import (
    load_evaluation_run, note_starts, evaluate_saved_charts,
    predict_teacher_forced, plot_chart_preview,
)

# All tunable values come from the one reviewed config module.
CONFIG = evaluation_config()
from donderformer.evaluation import (
    aggregate_overlapping_predictions,
    sample_chart,
    reconstruct_true_sequence,
    onset_metrics,
    joint_onset_type_metrics,
    score_from_counts,
    ochuman_score,
    generate_autoregressive,
    remove_adjacent_notes,
    to_binary_chart,
    paper_ochuman,
    extract_patterns,
    calculate_paper_metrics,
    calculate_f1_metrics
)

# Paste the folder printed by notebook 03, or use an absolute Path on PACE.
RUN_DIR = outputs_dir() / "runs" / "taikonation" / "<saved-run-folder>"
RUN_DIR = RUN_DIR.expanduser().resolve()
paths = DataPaths()
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
model, config, test_dataset = load_evaluation_run(RUN_DIR, paths, device)
print("Loaded", RUN_DIR, "on", device)
print("Data configuration:", config["data"])

if test_dataset.extractor.frame_ms != CONFIG["FRAME_MS"]:
    raise ValueError("CONFIG frame grid differs from this saved run")
context = load_reference_context(paths, test_dataset.split, test_dataset.extractor, CONFIG)


Pattern window units: PATTERN_WINDOWS_FRAMES is measured in 23 ms frames
(8 frames ≈ 0.18 s, 64 frames ≈ 1.5 s). The proposal does not fix the units, so frames
are used for now. Measure-based windows (as in Hanzen et al., 2025) would keep windows
comparable across tempos but require beat/measure information; switch later if the
team decides to.

Held-note counting: onset and joint precision/recall now count note starts only, so a
drumroll or denden counts once rather than once per frame. Earlier versions counted
held notes per frame; onset scores from those runs are not directly comparable.
Re-run evaluation on saved charts before comparing. Greedy matching and tolerance
are unchanged. The historical OCHuman-style recall pass and eight-frame paper
DCHuman/OCHuman/Over_PSpace/HI_PSpace definitions remain unchanged.


## Teacher-forced evaluation

Human note history is supplied to each test window. Four overlapping predictions are averaged per frame, then sampled with a shared NumPy generator seeded at 42 (batch size 64). Onset and joint onset/type scores use ±1 frame, greedy one-to-one matches, and micro counts across charts without matching across song boundaries. F1 counts held-note starts; the class histograms below still count frames.

The later OCHuman-style pass samples again using the same generator stream: it measures human onset recall within ±1 frame and allows a prediction to cover multiple human onsets. Its micro and macro averages are distinct from the paper-style OCHuman metric below. Run the sampling cells in order to reproduce the stream.


In [ ]:
EVAL_DIR = RUN_DIR / "eval"
TF_DIR = EVAL_DIR / "teacher_forced"
PLOTS_DIR = EVAL_DIR / "plots"
for d in (TF_DIR, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Evaluating", RUN_DIR.name)

TF_CHARTS_DIR = TF_DIR / "charts"
TF_CHARTS_DIR.mkdir(parents=True, exist_ok=True)


### Test Prediction Pipeline

**Trained Model + Test Audio/Note Windows**  
↓  
**Model Prediction**  
↓  
**Raw Scores (Logits)**  
↓  
**Softmax**  
↓  
**Probabilities for Each of the 7 Note Classes**  
↓  
**Used for Chart Generation**

In [ ]:
# Inference is performed song-by-song below, in small GPU batches.
model.eval()
INFERENCE_BATCH_SIZE = CONFIG["INFERENCE_BATCH_SIZE"]
print("Ready to evaluate", len(test_dataset.chart_ids), "test charts")

# Combining the overlapping predictions from different windows to produce one final 7-class probability distribution for each 23 ms timestep.

# Generating chart

Takes the final probability distribution for every 23 ms timestep and turns it into an actual generated chart.

In [ ]:
CLASS_NAMES = ["none", "don", "ka", "DON", "KA", "drumroll", "denden"]
rng = np.random.default_rng(CONFIG["TEACHER_FORCED_SEED"])



# Target vs. Model generated

# Onset evaluation

Regardless of note type like Don and Ka, this measures generated notes occur near the ground-truth note timings (within +- 23ms).

# Onset + type eval

In [ ]:


# Micro-averaged evaluation: accumulate counts across songs, never across boundaries.
model.eval()
total_true = total_pred = total_onset_matches = total_joint_matches = 0
true_class_counts = np.zeros(7, dtype=np.int64)
pred_class_counts = np.zeros(7, dtype=np.int64)

for song_index, song_id in enumerate(test_dataset.chart_ids, start=1):
    X, Y = test_dataset.chart_windows(song_id)
    if len(X) == 0:
        continue
    song_probs = []
    with torch.no_grad():
        for start in range(0, len(X), INFERENCE_BATCH_SIZE):
            batch = torch.from_numpy(
                X[start:start + INFERENCE_BATCH_SIZE].astype(np.float32)
            ).to(device)
            logits = model(batch)
            song_probs.append(torch.softmax(logits, dim=2).cpu().numpy())
    probs = np.concatenate(song_probs, axis=0)
    generated = sample_chart(aggregate_overlapping_predictions(probs), rng)
    true_sequence = reconstruct_true_sequence(Y)
    assert len(generated) == len(true_sequence)
    np.savez_compressed(TF_CHARTS_DIR / f"{song_id}.npz", human=true_sequence,
                        generated=generated, clean=remove_adjacent_notes(generated))

    _, _, _, onset_matches = onset_metrics(true_sequence, generated, tolerance=CONFIG["ONSET_TOLERANCE_FRAMES"])
    _, _, _, joint_matches = joint_onset_type_metrics(true_sequence, generated, tolerance=CONFIG["ONSET_TOLERANCE_FRAMES"])
    total_onset_matches += onset_matches
    total_joint_matches += joint_matches
    total_true += np.count_nonzero(note_starts(true_sequence))
    total_pred += np.count_nonzero(note_starts(generated))
    true_class_counts += np.bincount(true_sequence, minlength=7)
    pred_class_counts += np.bincount(generated, minlength=7)
    if song_index % CONFIG["PROGRESS_EVERY"] == 0:
        print(f"Evaluated {song_index}/{len(test_dataset.chart_ids)} charts")

precision, recall, f1 = score_from_counts(total_onset_matches, total_pred, total_true)
precision_type, recall_type, f1_type = score_from_counts(total_joint_matches, total_pred, total_true)
print(f"Onset: precision={precision:.4f}, recall={recall:.4f}, F1={f1:.4f}")
print(f"Onset + type: precision={precision_type:.4f}, recall={recall_type:.4f}, F1={f1_type:.4f}")
for name, actual, predicted in zip(CLASS_NAMES, true_class_counts, pred_class_counts):
    print(f"{name:10s} true={actual:8d} generated={predicted:8d}")

In [ ]:
results = {
    "onset_precision": float(precision),
    "onset_recall": float(recall),
    "onset_f1": float(f1),
    "joint_precision": float(precision_type),
    "joint_recall": float(recall_type),
    "joint_f1": float(f1_type)
}

import json

# Saved in the run's eval/teacher_forced/ folder
(TF_DIR / "results.json").write_text(json.dumps(results, indent=2))

print("Evaluation results saved to", TF_DIR / "results.json")
print(results)

# OCHuman-style Evaluation (±1 timestep)

In [ ]:
# Uses the already-trained model

import numpy as np
import torch



model.eval()

total_matched = 0
total_true = 0
song_scores = []
ochuman_groups = {}

for song_index, song_id in enumerate(test_dataset.chart_ids, start=1):

    X, Y = test_dataset.chart_windows(song_id)

    if len(X) == 0:
        continue

    song_probs = []

    with torch.no_grad():
        for start in range(0, len(X), INFERENCE_BATCH_SIZE):
            batch = torch.from_numpy(
                X[start:start + INFERENCE_BATCH_SIZE].astype(np.float32)
            ).to(device)

            logits = model(batch)
            probs = torch.softmax(logits, dim=2)
            song_probs.append(probs.cpu().numpy())

    probs = np.concatenate(song_probs, axis=0)

    generated = sample_chart(
        aggregate_overlapping_predictions(probs), rng
    )
    true_sequence = reconstruct_true_sequence(Y)

    assert len(generated) == len(true_sequence)

    matched, actual = ochuman_score(
        true_sequence,
        generated,
        tolerance=CONFIG["ONSET_TOLERANCE_FRAMES"]
    )

    total_matched += matched
    total_true += actual
    group = context.metadata[song_id]["difficulty"]
    entry = ochuman_groups.setdefault(group, dict(matched=0, actual=0, scores=[]))
    entry["matched"] += matched
    entry["actual"] += actual
    if actual:
        entry["scores"].append(matched / actual)

    if actual > 0:
        song_scores.append(matched / actual)

    if song_index % CONFIG["PROGRESS_EVERY"] == 0:
        print(f"Evaluated {song_index}/{len(test_dataset.chart_ids)} charts")


# Overall score (micro-average)
ochuman_micro = total_matched / total_true if total_true else 0

# Per-chart average (macro-average)
ochuman_macro = np.mean(song_scores) if song_scores else 0

print("\n===== OCHuman-style Results =====")
print(f"Micro-average: {ochuman_micro:.4f} ({ochuman_micro:.2%})")
print(f"Macro-average: {ochuman_macro:.4f} ({ochuman_macro:.2%})")
print(f"Charts evaluated: {len(song_scores)}")

# Save results separately
np.savez(
    TF_DIR / "ochuman.npz",
    micro=ochuman_micro,
    macro=ochuman_macro,
    song_scores=np.array(song_scores)
)

print("Results saved to", TF_DIR / "ochuman.npz")
# The legacy resampled recall pass also gets an overall/difficulty sidecar.
ochuman_breakdown = [dict(scope="overall", difficulty="", micro=float(ochuman_micro),
                          macro=float(ochuman_macro), charts=len(song_scores))]
for group, entry in sorted(ochuman_groups.items()):
    ochuman_breakdown.append(dict(scope="difficulty", difficulty=group,
        micro=entry["matched"] / entry["actual"] if entry["actual"] else None,
        macro=float(np.mean(entry["scores"])) if entry["scores"] else None,
        charts=len(entry["scores"])))
(TF_DIR / "ochuman_by_difficulty.json").write_text(json.dumps(ochuman_breakdown, indent=2))


In [ ]:
song_id = next(s for s, length in zip(test_dataset.chart_ids, test_dataset.lengths) if length > 0)
X, Y = test_dataset.chart_windows(song_id)
probs = predict_teacher_forced(model, X, device, CONFIG["INFERENCE_BATCH_SIZE"])
generated = sample_chart(aggregate_overlapping_predictions(probs), rng)
true_sequence = reconstruct_true_sequence(Y)
plot_chart_preview({"Human": true_sequence, "Teacher-forced": generated},
                   PLOTS_DIR / "human_vs_teacher_forced.png", CONFIG, CONFIG["PREVIEW_STEPS"])


## Autoregressive evaluation

Only audio is retained from each window. The previous 15 sampled notes are fed back; current sampling averages only predictions available so far, including the final three tail frames. The single-song preview uses `CONFIG["AR_SEED"]`. Full-test generation uses seed `CONFIG["AR_SEED"] + song_index` (one-based manifest order) and resumes saved chart sequences.

Raw and clean scores are computed per chart; summary.csv contains unweighted chart means. Cleaning tests adjacent notes in the original sequence. Paper metrics collapse held notes to their onsets, count empty-frame agreement for DCHuman/OCHuman, and use unique eight-frame binary patterns for Over/HI-PSpace. These definitions are preserved from notebook 03.

In [ ]:
song_id = next(s for s, length in zip(test_dataset.chart_ids, test_dataset.lengths) if length > 0)
X, Y = test_dataset.chart_windows(song_id)
true_sequence = reconstruct_true_sequence(Y)
generated_ar = generate_autoregressive(model, X, device, seed=CONFIG["AR_SEED"])
print("AR preview:", calculate_f1_metrics(true_sequence, generated_ar, CONFIG["ONSET_TOLERANCE_FRAMES"]))
plot_chart_preview({"Human": true_sequence, "Autoregressive": generated_ar},
                   PLOTS_DIR / "human_vs_autoregressive.png", CONFIG, CONFIG["PREVIEW_STEPS"])


In [ ]:
import time

SAVE_DIR = EVAL_DIR / "ar_test"
CHARTS_DIR = SAVE_DIR / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []
start_time = time.time()
for song_index, song_id in enumerate(test_dataset.chart_ids, start=1):
    save_path = CHARTS_DIR / f"{song_id}.npz"
    if not save_path.exists():
        X, Y = test_dataset.chart_windows(song_id)
        if len(X) == 0:
            continue
        human = reconstruct_true_sequence(Y)
        generated = generate_autoregressive(model, X, device, seed=CONFIG["AR_SEED"] + song_index)
        assert len(generated) == len(human)
        np.savez_compressed(save_path, human=human, generated=generated,
                            clean=remove_adjacent_notes(generated))
    if song_index % CONFIG["PROGRESS_EVERY"] == 0 or song_index == len(test_dataset.chart_ids):
        print(f"Generated/resumed {song_index}/{len(test_dataset.chart_ids)} charts")

# Model-independent scorer: Transformer outputs use this same function.
all_results = evaluate_saved_charts(CHARTS_DIR, context, SAVE_DIR)
print("Scored", len(all_results), "charts; legacy and extended metrics saved in", SAVE_DIR)


In [ ]:
song_id = next(s for s, length in zip(test_dataset.chart_ids, test_dataset.lengths) if length > 0)
with np.load(CHARTS_DIR / f"{song_id}.npz") as arrays:
    plot_chart_preview({"Human": arrays["human"], "AI - RAW": arrays["generated"], "AI - CLEAN": arrays["clean"]},
                       PLOTS_DIR / "taiko_chart_comparison.png", CONFIG)


## HI precision is the primary pattern metric

A pooled human reference can contain many more unique patterns than any single chart.
Coverage (`generated ∩ pool` divided by `pool`) can therefore be tiny even for a useful chart
and can rise simply because the chart is longer. HI **precision** instead asks: what share
of this chart's unique, non-empty generated patterns appears in the human pool?
It is a fraction from 0 to 1; all-zero windows are excluded from numerator and denominator,
and a chart with no non-empty windows gets null. `HI_PRIMARY="precision"` selects it for
headline tables, plots and retention. Set it to `"coverage"` to use coverage instead.

Existing `..._hi` coverage keys remain in per-chart JSON, unchanged in percent units.
New `..._hi_coverage` labels the same percentage explicitly; `..._hi_precision` is the new
fraction. `..._hi_primary` is the configured choice expressed as a fraction.
The legacy eight-frame paper `HI_PSpace` remains unchanged.


## Overall P-Space uses the matching human chart as its scale

The possible type-pattern space grows as `7**window`, so normalized percentages approach
zero at long windows. The headline Overall value is now
`unique generated patterns / unique matching-human patterns`: self-comparison is 1,
and fewer unique patterns gives a value below 1. Both variants report this ratio at every
window as `..._overall_ratio` (the existing `..._ratio` key is retained).

Summaries show normalized percentages only for the legacy eight-frame binary case.
Old normalized `..._overall` and `..._human_overall` keys remain in per-chart JSON for
compatibility, and are omitted from the new long-window summary/comparison tables.
Overall ratios retain empty windows; the empty-window exclusion applies to HI precision.


## Random density is excluded from comparisons

Random density is excluded because the control fixes the human note count; its active span
may still differ. Its density and in-band flag are omitted from random summary tables and
all notebook 07 comparison tests/tables; Don/Ka remains. Existing per-chart and seed JSON
density keys stay available for compatibility. `summary_notes.json` records the exclusion.


## Shared scoring and random control

The human pools include **all usable test charts** in the merged star group, including
charts missing from a model's generated directory. Binary and type-aware windows both
collapse held notes to starts. Null statistics are excluded with valid counts reported.
Density uses starts divided by elapsed seconds from first to last start; Don/Ka merges
small and big hits. Reference bands come only from the merged group's training charts.
`analysis/micro_summary.csv` additionally reports count-aggregate F1 overall and by group.
The legacy OCHuman resampling pass has `teacher_forced/ochuman_by_difficulty.json`.

The random control chooses the human number of distinct frames uniformly, then samples
from training start types in the merged group. Adjacent identical held types are excluded
conditionally so they cannot merge two starts. Metrics are averaged over seeds within chart.
Both raw and clean model results remain available; `PRIMARY_MODE`/`PRIMARY_VERSION` choose
the comparison. The legacy eight-frame paper metrics and exporter formats stay unchanged.


In [ ]:
tf_records = evaluate_saved_charts(TF_CHARTS_DIR, context, TF_DIR / "scored")
primary_records = all_results if CONFIG["PRIMARY_MODE"] == "ar" else tf_records
PRIMARY_ANALYSIS = (SAVE_DIR / "analysis" if CONFIG["PRIMARY_MODE"] == "ar"
                    else TF_DIR / "scored" / "analysis")
random_results = evaluate_random_baseline(context, PRIMARY_ANALYSIS / "random_baseline",
                                          chart_ids=[r["chart_id"] for r in primary_records])
print("Comparison inputs:", PRIMARY_ANALYSIS)
print("Open 07_compare_models.ipynb and set CONFIG['MODEL_RUNS'] in the shared config module.")


## Score existing charts without loading a model

Use this recipe independently in a fresh kernel (for LSTM, Transformer, or other generated frames).
It reads cached **labels only**, never a checkpoint or mel features. Restore the matching label cache
and raw TJA metadata first. `FeatureExtractor` must match the saved run's `data.extractor` settings
(or its legacy `data.features` ID); do not substitute another frame grid.

```python
from pathlib import Path
from donderformer.paths import DataPaths
from donderformer.data.feature_extractor import FeatureExtractor
from donderformer.evaluation_config import evaluation_config
from donderformer.chart_evaluation import load_reference_context, evaluate_random_baseline
from donderformer.evaluation import evaluate_saved_charts

config = evaluation_config()
# Read split/extractor from that run's config.json; no inference or retraining.
import json
run = Path("/absolute/path/to/saved/run")
saved = json.loads((run / "config.json").read_text())
extractor = FeatureExtractor(**saved["data"]["extractor"])
context = load_reference_context(DataPaths(), saved["data"]["split"], extractor, config)
root = run / "eval" / ("ar_test" if config["PRIMARY_MODE"] == "ar" else "teacher_forced")
output = root if config["PRIMARY_MODE"] == "ar" else root / "scored"
records = evaluate_saved_charts(root / "charts", context, output)
evaluate_random_baseline(context, output / "analysis" / "random_baseline",
                         chart_ids=[r["chart_id"] for r in records])
```

For legacy feature-ID-only configs, recover the exact settings from the matching
`data/features/<data.features>/config.json` rather than inventing defaults.
